# 9 — RhinoCompute STL + Volume Generation

This notebook does one thing only:

1. Send each propeller configuration to RhinoCompute.
2. Decode the selected mesh output, normally `MeshFinal`.
3. Save a binary STL file.
4. Compute mesh volume and write a CSV summary.

The workflow is intentionally minimal: no secondary CAD export, no external conversion backend, and no extra geometry-file format.


## 1. Imports

In [1]:
# Imports

import hashlib
import json
import struct
import threading
import time
from http.server import HTTPServer, SimpleHTTPRequestHandler
from pathlib import Path

import pandas as pd
import rhino3dm
from compute_rhino3d import Grasshopper, Util
from tqdm.auto import tqdm


c:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset\.venv311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configuration

Keep the settings here. Under normal use, you should only need to change `RHINO_COMPUTE_URL`, `MAX_CONFIGS`, or the CSV candidates.


In [2]:
# Configuration

# =============================================================================
# 1) RHINOCOMPUTE
# =============================================================================

# Use the port where your RhinoCompute server is actually running.
# Common values are:
#   - http://localhost:5000/
#   - http://localhost:6500/
#   - http://localhost:8081/
RHINO_COMPUTE_URL = "http://localhost:5000/"
RHINO_COMPUTE_API_KEY = ""   # leave empty if no key is required


# =============================================================================
# 2) LOCAL FILE SERVER FOR propeller.gh
# =============================================================================

GH_SERVER_PORT = 9876


# =============================================================================
# 3) PROJECT ROOT / PATHS
# =============================================================================

# Keep True unless you intentionally want to force BASE_DIR manually.
AUTO_DETECT_BASE_DIR = True

# If AUTO_DETECT_BASE_DIR=False, this folder is used directly.
BASE_DIR = Path.cwd().resolve()

GH_FILENAME = "propeller.gh"

# The notebook will use the first existing CSV in this list.
PARAMS_CSV_CANDIDATES = [
    "representative_geometry.csv",
    "prop_geometrical_params.csv",
]

# Output CSV written into BASE_DIR/csv/.
RESULTS_CSV_NAME = "rhino_stl_volume_generation.csv"


# =============================================================================
# 4) GRASSHOPPER PARAMETERS
# =============================================================================

EXPORT_SOLUTION = True
POSITION_OFFSET = 50

# Preferred Grasshopper output containing the final mesh.
# Your current .gh appears to expose this as MeshFinal.
GH_OUTPUT_MESH_PARAM = "MeshFinal"

# If True, the notebook scans other Grasshopper outputs when MeshFinal cannot be decoded.
AUTO_DETECT_MESH_OUTPUT = True


# =============================================================================
# 5) BATCH / DEBUG SETTINGS
# =============================================================================

MAX_CONFIGS = None                  # None = process all rows
INTER_CALL_DELAY_S = 0.0

# This proves that one STL is really written before launching the full batch.
RUN_SINGLE_TEST_BEFORE_BATCH = True
TEST_CONFIG_INDEX = 0

# Save raw RhinoCompute JSON when no mesh can be decoded.
SAVE_DEBUG_JSON = True

# Output folders.
STL_FOLDER_NAME = "stl"
DEBUG_FOLDER_NAME = "debug"


## 3. Input validation and paths

In [3]:
# Input validation and path setup

def _find_project_root(start_dir, gh_filename, csv_candidates):
    """Find a folder containing propeller.gh and csv/<params>.csv."""
    start_dir = Path(start_dir).resolve()
    search_roots = [start_dir] + list(start_dir.parents)

    # Search upward first.
    for root in search_roots:
        gh_ok = (root / gh_filename).exists()
        csv_ok = any((root / "csv" / name).exists() for name in csv_candidates)
        if gh_ok and csv_ok:
            return root

    # Then search one/two levels downward from the current folder.
    # This helps when Jupyter was launched from a parent directory.
    for depth_pattern in ["*", "*/*"]:
        for child in start_dir.glob(depth_pattern):
            if not child.is_dir():
                continue
            gh_ok = (child / gh_filename).exists()
            csv_ok = any((child / "csv" / name).exists() for name in csv_candidates)
            if gh_ok and csv_ok:
                return child.resolve()

    return start_dir


if AUTO_DETECT_BASE_DIR:
    BASE_DIR = _find_project_root(BASE_DIR, GH_FILENAME, PARAMS_CSV_CANDIDATES)
else:
    BASE_DIR = Path(BASE_DIR).resolve()

CSV_DIR   = BASE_DIR / "csv"
STL_DIR   = BASE_DIR / STL_FOLDER_NAME
DEBUG_DIR = BASE_DIR / DEBUG_FOLDER_NAME

GH_FILE = BASE_DIR / GH_FILENAME

PARAMS_CSV = None
for name in PARAMS_CSV_CANDIDATES:
    candidate = CSV_DIR / name
    if candidate.exists():
        PARAMS_CSV = candidate
        break

if PARAMS_CSV is None:
    raise FileNotFoundError(
        "No input geometry CSV found. Tried:\n" +
        "\n".join(str(CSV_DIR / name) for name in PARAMS_CSV_CANDIDATES)
    )

RESULTS_CSV = CSV_DIR / RESULTS_CSV_NAME

if not GH_FILE.exists():
    raise FileNotFoundError(
        f"Grasshopper definition not found: {GH_FILE}\n"
        "Make sure propeller.gh is in the project root."
    )

for folder in [CSV_DIR, STL_DIR, DEBUG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

params_df = pd.read_csv(PARAMS_CSV)

if MAX_CONFIGS is not None:
    params_df = params_df.head(MAX_CONFIGS).copy()

print("Project root          :", BASE_DIR)
print("Grasshopper definition:", GH_FILE, f"(MD5: {hashlib.md5(GH_FILE.read_bytes()).hexdigest()})")
print("Input CSV             :", PARAMS_CSV)
print("Configurations        :", len(params_df))
print("STL output directory  :", STL_DIR)
print("Results CSV           :", RESULTS_CSV)
print("RhinoCompute URL      :", RHINO_COMPUTE_URL)


Project root          : C:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset
Grasshopper definition: C:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset\propeller.gh (MD5: d0fd3634cb0a428e36972da5585cdf4c)
Input CSV             : C:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset\csv\representative_geometry.csv
Configurations        : 100
STL output directory  : C:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset\stl
Results CSV           : C:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset\csv\rhino_stl_volume_generation.csv
RhinoCompute URL      : http://localhost:5000/


## 4. Start local file server

RhinoCompute fetches `propeller.gh` through this local HTTP server.


In [4]:
# Start local HTTP server to serve propeller.gh

class _QuietHandler(SimpleHTTPRequestHandler):
    """SimpleHTTPRequestHandler with logging suppressed."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, directory=str(BASE_DIR), **kwargs)

    def log_message(self, format, *args):  # noqa: A002
        pass


class _ReusableHTTPServer(HTTPServer):
    allow_reuse_address = True


# If this cell is re-run, shut down the previous server first.
try:
    _server.shutdown()
    _server.server_close()
    time.sleep(0.2)
except NameError:
    pass
except Exception as exc:
    print(f"Previous server could not be closed cleanly: {exc}")

_server = _ReusableHTTPServer(("localhost", GH_SERVER_PORT), _QuietHandler)
_server_thread = threading.Thread(target=_server.serve_forever, daemon=True)
_server_thread.start()

GH_URL = f"http://localhost:{GH_SERVER_PORT}/{GH_FILE.name}"
print("File server started:", GH_URL)
print("Open this URL in a browser if you want to confirm that propeller.gh is being served.")


File server started: http://localhost:9876/propeller.gh
Open this URL in a browser if you want to confirm that propeller.gh is being served.


## 5. Helper functions

In [5]:
# Helper functions

# ---------------------------------------------------------------------------
# CSV header -> Grasshopper input parameter name
# ---------------------------------------------------------------------------
CSV_TO_GH = {
    "radius":          "impellerRadius",
    "ring height":     "impellerHeight",
    "ring thickness":  "impellerThickness",
    "blade count":     "bladeCount",
    "inner thickness": "innerThickness",
    "inner max pos":   "innerMaxPos",
    "inner camber":    "innerCamber",
    "inner chord":     "innerChord",
    "inner angle":     "innerAngle",
    "mid radial pos":  "middlePos",
    "mid chord":       "middleChord",
    "mid angle":       "middleAngle",
    "outer thickness": "outerThickness",
    "outer max pos":   "outerMaxPos",
    "outer camber":    "outerCamber",
    "outer chord":     "outerChord",
    "outer angle":     "outerAngle",
}


def row_to_gh_inputs(row):
    """Convert a CSV row to Grasshopper input names and values."""
    missing = [c for c in CSV_TO_GH if c not in row.index]
    if missing:
        raise KeyError(f"Missing required geometry columns in CSV: {missing}")

    inputs = {gh_name: float(row[csv_col]) for csv_col, gh_name in CSV_TO_GH.items()}
    inputs["positionOffset"] = int(POSITION_OFFSET)
    inputs["exportSolution"] = bool(EXPORT_SOLUTION)
    return inputs


def call_rhinocompute(gh_url, inputs):
    """Evaluate the Grasshopper definition through RhinoCompute."""
    trees = []
    for name, value in inputs.items():
        tree = Grasshopper.DataTree(name)
        tree.Append([0], [value])
        trees.append(tree)
    return Grasshopper.EvaluateDefinition(gh_url, trees)


def _json_loads_repeated(value, max_depth=4):
    """Decode JSON strings that may be nested/double-encoded."""
    current = value
    for _ in range(max_depth):
        if not isinstance(current, str):
            break
        s = current.strip()
        try:
            current = json.loads(s)
        except Exception:
            break
    return current


def _as_mesh_list(obj):
    """Return all rhino3dm.Mesh objects found inside obj/list/tuple."""
    meshes = []
    if obj is None:
        return meshes

    if isinstance(obj, rhino3dm.Mesh):
        return [obj]

    if isinstance(obj, (list, tuple)):
        for sub in obj:
            meshes.extend(_as_mesh_list(sub))
        return meshes

    return meshes


def _decode_item_to_meshes(raw):
    """Decode one RhinoCompute item['data'] field into zero or more meshes."""
    meshes = []
    if raw is None:
        return meshes

    # 1) Draco base64 path. RhinoCompute often returns this as a quoted string.
    if isinstance(raw, str):
        s = raw.strip()
        if len(s) >= 2 and s[0] == '"' and s[-1] == '"':
            s_unquoted = s[1:-1]
        else:
            s_unquoted = s

        if s_unquoted.startswith("RFJBQ08") or s_unquoted.startswith("DRACO"):
            try:
                obj = rhino3dm.DracoCompression.DecompressBase64String(s_unquoted)
                meshes.extend(_as_mesh_list(obj))
            except Exception:
                pass

    if meshes:
        return meshes

    # 2) OpenNURBS JSON path.
    decoded = _json_loads_repeated(raw)
    try:
        obj = rhino3dm.CommonObject.Decode(decoded)
        meshes.extend(_as_mesh_list(obj))
    except Exception:
        pass

    return meshes


def _response_messages(response_json):
    messages = []
    for key in ["errors", "warnings"]:
        for item in response_json.get(key, []) or []:
            if isinstance(item, dict):
                messages.append(item.get("Message") or item.get("message") or str(item))
            else:
                messages.append(str(item))
    return messages


def _output_summaries(response_json):
    summaries = []
    for v in response_json.get("values", []) or []:
        name = v.get("ParamName", "<unnamed>")
        count = sum(len(items) for items in (v.get("InnerTree", {}) or {}).values())
        summaries.append(f"{name}({count})")
    return summaries


def decode_meshes_from_response(response_json, preferred_output=GH_OUTPUT_MESH_PARAM):
    """
    Decode meshes from the preferred Grasshopper output.
    If AUTO_DETECT_MESH_OUTPUT=True, it also scans all other outputs.
    """
    values = response_json.get("values", []) or []
    messages = _response_messages(response_json)
    all_output_names = [v.get("ParamName", "<unnamed>") for v in values]

    ordered_outputs = []
    preferred = [v for v in values if v.get("ParamName") == preferred_output]
    others = [v for v in values if v.get("ParamName") != preferred_output]
    ordered_outputs.extend(preferred)
    if AUTO_DETECT_MESH_OUTPUT:
        ordered_outputs.extend(others)

    for output in ordered_outputs:
        output_name = output.get("ParamName", "<unnamed>")
        meshes = []
        for branch_items in (output.get("InnerTree", {}) or {}).values():
            for item in branch_items:
                meshes.extend(_decode_item_to_meshes(item.get("data")))
        if meshes:
            return meshes, messages, all_output_names, output_name

    return [], messages, all_output_names, ""


def save_debug_response(config_id, response_json):
    """Save raw RhinoCompute response for troubleshooting."""
    if not SAVE_DEBUG_JSON:
        return None
    path = DEBUG_DIR / f"rhinocompute_response_config_{config_id}.json"
    try:
        path.write_text(json.dumps(response_json, indent=2), encoding="utf-8")
        return path
    except Exception as exc:
        print(f"Could not save debug JSON for config {config_id}: {exc}")
        return None


def _prepare_mesh(mesh):
    """Triangulate one mesh in-place and return it."""
    mesh.Faces.ConvertQuadsToTriangles()
    try:
        mesh.Normals.ComputeNormals()
    except Exception:
        pass
    try:
        mesh.Compact()
    except Exception:
        pass
    return mesh


def _safe_len(collection):
    """
    Robust length helper for rhino3dm collections.

    Some rhino3dm collection classes do not expose RhinoCommon-style .Count.
    In Python, len(collection) is usually the correct API.
    """
    try:
        return len(collection)
    except Exception:
        pass

    for attr_name in ("Count", "count"):
        try:
            attr = getattr(collection, attr_name)
            return int(attr() if callable(attr) else attr)
        except Exception:
            pass

    try:
        return sum(1 for _ in collection)
    except Exception as exc:
        raise TypeError(f"Could not determine collection length for {type(collection)}") from exc


def _get_vertex_xyz(vertex):
    """Return a vertex as plain float coordinates."""
    if hasattr(vertex, "X") and hasattr(vertex, "Y") and hasattr(vertex, "Z"):
        return float(vertex.X), float(vertex.Y), float(vertex.Z)
    return float(vertex[0]), float(vertex[1]), float(vertex[2])


def _face_indices(face):
    """
    Return mesh-face vertex indices.

    Supports both possible rhino3dm access patterns:
    - face[0], face[1], face[2], face[3]
    - face.A, face.B, face.C, face.D
    """
    try:
        a = int(face[0])
        b = int(face[1])
        c = int(face[2])
        try:
            d = int(face[3])
        except Exception:
            d = c
        return a, b, c, d
    except Exception:
        a = int(face.A)
        b = int(face.B)
        c = int(face.C)
        d = int(getattr(face, "D", c))
        return a, b, c, d


def _iter_faces(faces):
    """Iterate over a rhino3dm MeshFaceList without assuming .Count exists."""
    try:
        n = _safe_len(faces)
        for i in range(n):
            yield faces[i]
    except Exception:
        for face in faces:
            yield face


def _iter_mesh_triangles(mesh):
    """
    Yield triangles as vertex-index triples.

    Quads are split defensively. In normal use _prepare_mesh() converts quads
    before this function is called, so almost all faces should already be triangles.
    """
    vertex_count = _safe_len(mesh.Vertices)

    for face in _iter_faces(mesh.Faces):
        a, b, c, d = _face_indices(face)

        # Basic sanity check: skip obviously invalid faces rather than writing corrupt STL.
        if min(a, b, c) < 0 or max(a, b, c) >= vertex_count:
            continue

        if d == c or d < 0 or d >= vertex_count:
            yield a, b, c
        else:
            yield a, b, c
            yield a, c, d


def _mesh_triangle_count(mesh):
    return sum(1 for _ in _iter_mesh_triangles(mesh))


def _triangle_volume_from_origin(p0, p1, p2):
    """Signed tetrahedron volume contribution for triangle p0-p1-p2."""
    x0, y0, z0 = p0
    x1, y1, z1 = p1
    x2, y2, z2 = p2
    return (
        x0 * (y1 * z2 - y2 * z1)
        + x1 * (y2 * z0 - y0 * z2)
        + x2 * (y0 * z1 - y1 * z0)
    ) / 6.0


def mesh_list_to_stl_and_volume(meshes, filepath):
    """
    Write one or more rhino3dm meshes to a binary STL and compute volume.

    Volume is computed from the triangle mesh using signed tetrahedra.
    It is reliable only when the mesh is closed and consistently oriented.
    For multiple mesh pieces, absolute volume is accumulated per mesh to avoid
    cancellation if different parts have opposite orientation.
    """
    filepath = Path(filepath)
    filepath.parent.mkdir(parents=True, exist_ok=True)

    prepared = [_prepare_mesh(mesh) for mesh in meshes]
    total_vertices = sum(_safe_len(mesh.Vertices) for mesh in prepared)
    total_triangles = sum(_mesh_triangle_count(mesh) for mesh in prepared)

    if total_vertices == 0 or total_triangles == 0:
        raise ValueError(
            f"Decoded mesh is empty or invalid: {total_vertices} vertices, "
            f"{total_triangles} triangles"
        )

    volume_mm3 = 0.0

    with filepath.open("wb") as f:
        header = b"Generated by 9_rhino_stl_generation_STL_VOLUME_ONLY.ipynb"
        f.write(header[:80].ljust(80, b"\0"))
        f.write(struct.pack("<I", total_triangles))

        for mesh in prepared:
            vertices = mesh.Vertices
            mesh_signed_volume = 0.0

            for a, b, c in _iter_mesh_triangles(mesh):
                p0 = _get_vertex_xyz(vertices[a])
                p1 = _get_vertex_xyz(vertices[b])
                p2 = _get_vertex_xyz(vertices[c])

                x0, y0, z0 = p0
                x1, y1, z1 = p1
                x2, y2, z2 = p2

                ux, uy, uz = x1 - x0, y1 - y0, z1 - z0
                vx, vy, vz = x2 - x0, y2 - y0, z2 - z0

                nx = uy * vz - uz * vy
                ny = uz * vx - ux * vz
                nz = ux * vy - uy * vx

                length = (nx**2 + ny**2 + nz**2) ** 0.5
                if length > 0:
                    nx, ny, nz = nx / length, ny / length, nz / length
                else:
                    nx, ny, nz = 0.0, 0.0, 0.0

                f.write(struct.pack("<fff", float(nx), float(ny), float(nz)))
                f.write(struct.pack("<fff", x0, y0, z0))
                f.write(struct.pack("<fff", x1, y1, z1))
                f.write(struct.pack("<fff", x2, y2, z2))
                f.write(struct.pack("<H", 0))

                mesh_signed_volume += _triangle_volume_from_origin(p0, p1, p2)

            volume_mm3 += abs(mesh_signed_volume)

    size = filepath.stat().st_size if filepath.exists() else 0
    if size <= 84:
        raise IOError(f"STL was not written correctly: {filepath} (size={size} bytes)")

    return {
        "volume_mm3": volume_mm3,
        "n_meshes": len(prepared),
        "n_vertices": total_vertices,
        "n_triangles": total_triangles,
        "file_size_bytes": size,
    }


def generate_stl_for_row(row):
    """Generate STL and volume for a single CSV row and return a result record."""
    config_id = int(row["config_id"])
    stl_path = STL_DIR / f"prop_{config_id}.stl"

    record = {
        "config_id": config_id,
        "geometry_ok": False,
        "stl_ok": False,
        "stl_path": str(stl_path.relative_to(BASE_DIR)),
        "stl_abs_path": str(stl_path),
        "stl_size_bytes": None,
        "volume_mm3": None,
        "volume_cm3": None,
        "volume_m3": None,
        "n_meshes": None,
        "n_vertices": None,
        "n_triangles": None,
        "selected_output_name": "",
        "all_output_names": "",
        "output_summary": "",
        "error_message": "",
        "debug_json_path": "",
    }

    try:
        inputs = row_to_gh_inputs(row)
        response = call_rhinocompute(GH_URL, inputs)
        record["output_summary"] = "; ".join(_output_summaries(response))

        meshes, messages, output_names, selected_output = decode_meshes_from_response(response)
        record["all_output_names"] = "; ".join(output_names)
        record["selected_output_name"] = selected_output

        if not meshes:
            debug_path = save_debug_response(config_id, response)
            if debug_path is not None:
                record["debug_json_path"] = str(debug_path.relative_to(BASE_DIR))
            record["error_message"] = (
                f"No decodable mesh found. Preferred output={GH_OUTPUT_MESH_PARAM!r}. "
                f"Outputs: {record['output_summary']}. "
                f"RhinoCompute messages: {messages}."
            )
            return record

        stats = mesh_list_to_stl_and_volume(meshes, stl_path)
        record.update({
            "geometry_ok": True,
            "stl_ok": True,
            "stl_size_bytes": stats["file_size_bytes"],
            "volume_mm3": round(stats["volume_mm3"], 2),
            "volume_cm3": round(stats["volume_mm3"] / 1e3, 4),
            "volume_m3": round(stats["volume_mm3"] / 1e9, 9),
            "n_meshes": stats["n_meshes"],
            "n_vertices": stats["n_vertices"],
            "n_triangles": stats["n_triangles"],
        })

    except Exception as exc:
        record["error_message"] = str(exc)

    return record


## 6. Configure RhinoCompute client

In [6]:
# Configure compute_rhino3d client

Util.url = RHINO_COMPUTE_URL
Util.apiKey = RHINO_COMPUTE_API_KEY

print("compute_rhino3d configured:", RHINO_COMPUTE_URL)


compute_rhino3d configured: http://localhost:5000/


## 7. Single test + batch STL generation

The first test writes one STL and checks that the file exists before the full batch starts. This prevents silent runs that generate zero files.


In [7]:
# Single test + batch generation

if RUN_SINGLE_TEST_BEFORE_BATCH:
    print("=" * 80)
    print("Single test before batch")
    test_row = params_df.iloc[TEST_CONFIG_INDEX]
    test_record = generate_stl_for_row(test_row)

    stl_path = Path(test_record["stl_abs_path"])

    print("config_id       :", test_record["config_id"])
    print("selected output :", test_record["selected_output_name"])
    print("outputs         :", test_record["output_summary"])
    print("STL path        :", test_record["stl_abs_path"])
    print("STL exists      :", stl_path.exists())
    print("STL size bytes  :", stl_path.stat().st_size if stl_path.exists() else 0)
    print("volume mm3      :", test_record["volume_mm3"])
    print("volume cm3      :", test_record["volume_cm3"])
    if test_record["error_message"]:
        print("ERROR           :", test_record["error_message"])
    if test_record["debug_json_path"]:
        print("Debug JSON      :", BASE_DIR / test_record["debug_json_path"])
    print("=" * 80)

    if not test_record["stl_ok"]:
        raise RuntimeError(
            "The one-configuration test did not write an STL. "
            "Batch stopped so you do not silently get zero files. "
            "Check the printed output and debug JSON above."
        )


results = []

for _, row in tqdm(params_df.iterrows(), total=len(params_df), desc="Generating STL files"):
    record = generate_stl_for_row(row)
    results.append(record)

    if INTER_CALL_DELAY_S > 0:
        time.sleep(INTER_CALL_DELAY_S)

results_df = pd.DataFrame(results)

n_geom = int(results_df["geometry_ok"].sum())
n_stl = int(results_df["stl_ok"].sum())
n_fail = len(results_df) - n_geom

print("\nDone")
print("  geometries succeeded:", n_geom)
print("  geometries failed   :", n_fail)
print("  STL files written   :", n_stl)
print("  STL folder          :", STL_DIR)


Single test before batch
config_id       : 132
selected output : MeshFinal
outputs         : MeshLauncher(1); MeshRing(1); MeshProfile(5); MeshFinal(1); InnerProfile(1); MiddleProfile(1); OuterProfile(1); MeshSimpleInterface(1); MeshUntrimmedBlades(5)
STL path        : C:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset\stl\prop_132.stl
STL exists      : True
STL size bytes  : 2559284
volume mm3      : 31912.95
volume cm3      : 31.9129


Generating STL files: 100%|██████████| 100/100 [10:41<00:00,  6.41s/it]


Done
  geometries succeeded: 100
  geometries failed   : 0
  STL files written   : 100
  STL folder          : C:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset\stl


## 8. Export results CSV

In [8]:
# Export results CSV

if "results_df" not in globals():
    raise RuntimeError("results_df does not exist. Run the single test + batch generation cell first.")

results_df.to_csv(RESULTS_CSV, index=False)
print("Results written to:", RESULTS_CSV)
results_df.head(10)


Results written to: C:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset\csv\rhino_stl_volume_generation.csv


,config_id,geometry_ok,stl_ok,stl_path,stl_abs_path,stl_size_bytes,volume_mm3,volume_cm3,volume_m3,n_meshes,n_vertices,n_triangles,selected_output_name,all_output_names,output_summary,error_message,debug_json_path
0,132,True,True,stl\prop_132.stl,C:\Users\hecto\Desktop\bachelor-thesis\IDEAL-P...,2559284,31912.95,31.9129,0.000032,1,28849,51184,MeshFinal,MeshLauncher; MeshRing; MeshProfile; MeshFinal...,MeshLauncher(1); MeshRing(1); MeshProfile(5); ...,,
1,156,True,True,stl\prop_156.stl,C:\Users\hecto\Desktop\bachelor-thesis\IDEAL-P...,1728684,8551.44,8.5514,0.000009,1,19959,34572,MeshFinal,MeshLauncher; MeshRing; MeshProfile; MeshFinal...,MeshLauncher(1); MeshRing(1); MeshProfile(4); ...,,
2,168,True,True,stl\prop_168.stl,C:\Users\hecto\Desktop\bachelor-thesis\IDEAL-P...,2496484,14356.79,14.3568,0.000014,1,28418,49928,MeshFinal,MeshLauncher; MeshRing; MeshProfile; MeshFinal...,MeshLauncher(1); MeshRing(1); MeshProfile(5); ...,,
3,192,True,True,stl\prop_192.stl,C:\Users\hecto\Desktop\bachelor-thesis\IDEAL-P...,2489884,27243.94,27.2439,0.000027,1,27917,49796,MeshFinal,MeshLauncher; MeshRing; MeshProfile; MeshFinal...,MeshLauncher(1); MeshRing(1); MeshProfile(5); ...,,
4,299,True,True,stl\prop_299.stl,C:\Users\hecto\Desktop\bachelor-thesis\IDEAL-P...,1870184,11973.26,11.9733,0.000012,1,21581,37402,MeshFinal,MeshLauncher; MeshRing; MeshProfile; MeshFinal...,MeshLauncher(1); MeshRing(1); MeshProfile(4); ...,,
5,329,True,True,stl\prop_329.stl,C:\Users\hecto\Desktop\bachelor-thesis\IDEAL-P...,1711184,15327.34,15.3273,0.000015,1,19615,34222,MeshFinal,MeshLauncher; MeshRing; MeshProfile; MeshFinal...,MeshLauncher(1); MeshRing(1); MeshProfile(4); ...,,
6,356,True,True,stl\prop_356.stl,C:\Users\hecto\Desktop\bachelor-thesis\IDEAL-P...,1599484,9863.06,9.8631,0.000010,1,18281,31988,MeshFinal,MeshLauncher; MeshRing; MeshProfile; MeshFinal...,MeshLauncher(1); MeshRing(1); MeshProfile(3); ...,,
7,406,True,True,stl\prop_406.stl,C:\Users\hecto\Desktop\bachelor-thesis\IDEAL-P...,2078984,28510.81,28.5108,0.000029,1,23386,41578,MeshFinal,MeshLauncher; MeshRing; MeshProfile; MeshFinal...,MeshLauncher(1); MeshRing(1); MeshProfile(4); ...,,
8,427,True,True,stl\prop_427.stl,C:\Users\hecto\Desktop\bachelor-thesis\IDEAL-P...,2539084,15512.37,15.5124,0.000016,1,29042,50780,MeshFinal,MeshLauncher; MeshRing; MeshProfile; MeshFinal...,MeshLauncher(1); MeshRing(1); MeshProfile(6); ...,,
9,455,True,True,stl\prop_455.stl,C:\Users\hecto\Desktop\bachelor-thesis\IDEAL-P...,2333084,23246.05,23.2460,0.000023,1,26183,46660,MeshFinal,MeshLauncher; MeshRing; MeshProfile; MeshFinal...,MeshLauncher(1); MeshRing(1); MeshProfile(4); ...,,


## 9. Summary

In [9]:
# Summary

ok_df = results_df[results_df["geometry_ok"]]
fail_df = results_df[~results_df["geometry_ok"]]

print("=" * 80)
print(f"Total configurations : {len(results_df)}")
print(f"Successful geometry  : {len(ok_df)}")
print(f"Failed geometry      : {len(fail_df)}")
print(f"STL files written    : {int(results_df['stl_ok'].sum())}")
print(f"STL directory        : {STL_DIR}")
print(f"Results CSV          : {RESULTS_CSV}")

if len(ok_df) > 0:
    print()
    print("Volume statistics for successful configs:")
    print(f"  Mean mm³   : {ok_df['volume_mm3'].mean():,.1f}")
    print(f"  Median mm³ : {ok_df['volume_mm3'].median():,.1f}")
    print(f"  Min mm³    : {ok_df['volume_mm3'].min():,.1f}")
    print(f"  Max mm³    : {ok_df['volume_mm3'].max():,.1f}")
    print(f"  Mean cm³   : {ok_df['volume_cm3'].mean():,.4f}")
    print()
    print("Mesh statistics:")
    print(f"  Avg meshes    : {ok_df['n_meshes'].mean():,.1f}")
    print(f"  Avg triangles : {ok_df['n_triangles'].mean():,.0f}")
    print(f"  Avg vertices  : {ok_df['n_vertices'].mean():,.0f}")

if len(fail_df) > 0:
    print()
    print(f"First {min(10, len(fail_df))} failed config IDs:")
    for _, r in fail_df.head(10).iterrows():
        print(f"  config_id={int(r['config_id'])}: {str(r['error_message'])[:220]}")
        if str(r.get("debug_json_path", "")):
            print(f"    debug: {BASE_DIR / r['debug_json_path']}")

print("=" * 80)


Total configurations : 100
Successful geometry  : 100
Failed geometry      : 0
STL files written    : 100
STL directory        : C:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset\stl
Results CSV          : C:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset\csv\rhino_stl_volume_generation.csv

Volume statistics for successful configs:
  Mean mm³   : 17,684.3
  Median mm³ : 16,941.9
  Min mm³    : 7,018.2
  Max mm³    : 34,705.2
  Mean cm³   : 17.6843

Mesh statistics:
  Avg meshes    : 1.0
  Avg triangles : 41,392
  Avg vertices  : 23,668
